Dataset Description & Use Case
Dataset Overview:
This dataset contains transactional records of customer purchases, 
 detailing the transaction ID, customer, date, amount, currency, and status. 
 It contains 11 records in total.

Business Use Case:
This data is typically used for revenue tracking, customer lifetime value (CLV) analysis, and sales forecasting. 
However, if the transaction records are messy, 
any downstream revenue dashboards or behavioral models will output skewed and inaccurate results.

In [4]:
import pandas as pd

df = pd.read_csv('C:\\Users\\mohit\\Downloads\\week2_customer_transactions_messy.csv')

print("Dataset Shape:", df.shape)
print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\nSample of Inconsistent Formatting")
print(df[['transaction_date', 'currency', 'payment_method']].head(5))

Dataset Shape: (11, 9)

--- Missing Values ---
transaction_id      0
customer_id         1
transaction_date    0
amount              1
currency            0
payment_method      1
status              0
region              0
last_updated        1
dtype: int64

Sample of Inconsistent Formatting
  transaction_date currency payment_method
0       2026-01-05      EUR           card
1       2026/01/06      EUR           CARD
2       06-01-2026      USD  bank_transfer
3       2026-01-07      EUR           card
4       2026-01-07     EURO           cash


In [5]:
total_rows = len(df)

# Completeness Rate (Percentage of rows with NO missing values)
complete_rows = df.dropna().shape[0]
completeness_rate = (complete_rows / total_rows) * 100

# Duplication Rate (Percentage of rows involved in duplicate transaction_ids)
duplicates = df.duplicated(subset=['transaction_id'], keep=False).sum()
duplication_rate = (duplicates / total_rows) * 100

# Error Rate (Percentage of invalid transactions)
error_rows = df[df['amount'] < 0].shape[0]
error_rate = (error_rows / total_rows) * 100

print(f"Completeness Rate: {completeness_rate:.2f}% ({complete_rows}/{total_rows} rows)")
print(f"Duplication Rate: {duplication_rate:.2f}% ({duplicates} rows affected)")
print(f"Error Rate (Negative Amounts): {error_rate:.2f}% ({error_rows} rows affected)")

Completeness Rate: 72.73% (8/11 rows)
Duplication Rate: 18.18% (2 rows affected)
Error Rate (Negative Amounts): 9.09% (1 rows affected)


Validation Rules
To automatically detect problematic records in future pipelines, 
we define the following validation rules:
Validity Rule (Amount): amount must be >= 0. 
Negative transactions (-35.00) indicate data entry errors or unstructured refund logic.
Uniqueness Rule (TransactionID): transaction_id must be exactly 1 per record. 
No exact duplicates are allowed, as this inflates gross revenue.
Completeness Rule (Critical Columns): Both customer_id and amount must not be null. 
An orphaned transaction without a customer or a price cannot be used for analytics.

Data Quality Audit Summary & FindingsBased on the Python analysis above, 
here is the structured audit of the dataset:
Data DimensionIssue TypeAffected RowsSeverityRecommended Next ActionCompletenessNull values in 
customer_id, amount, payment_method, and last_updated3 rows total (8/11 complete)
HighCheck if missing IDs/amounts can be recovered; otherwise, drop them for strictly user-level analysis.
UniquenessDuplicate transaction_ids2 rowsCriticalDrop the exact duplicate, keeping the first valid occurrence.
ValidityNegative amount values (-35.00)1 rowMediumInvestigate if this is a refund. 
Convert to absolute value if it is a typo, or filter out of gross sales.
ConsistencyDate formats (2026-01-05 vs 2026/01/06 vs 06-01-2026) & Text formats (EUR vs EURO, card vs CARD)Multiple rowsLow

Recommended Cleaning Actions for Next Week
For the upcoming data cleaning chapter, I recommend we apply the following structured steps (without implementing them just yet):

Standardize Strings & Dates: Convert transaction_date to pd.to_datetime() to fix the mixed delimiters. Convert payment_method, currency, and region to .str.lower() and map euro to eur.

Deduplicate: Apply df.drop_duplicates(subset=['transaction_id'], keep='first') to clean the primary keys.

Handle Missing/Invalid Data: Impute or drop the row with the missing amount, and isolate the row with the negative amount into a separate "needs review" dataframe.